In [78]:
import pandas as pd 

hayek_citations_df = pd.read_csv("../data/processed/hayek_citations.csv")
mises_citations_df = pd.read_csv("../data/processed/mises_citations.csv")

In [79]:
mises_citations_df.head()

,Unnamed: 0,paper_id,raw,context,co_cited_count,section_id,paragraph_id,sentence_id,sentence_seq_number,reference_seq_number,...,sentence_count,reference_count,source title,similarity,keywords,human_action_chapter_by_ref_page,human_action_chapter_number,human_action_chapter_name,human_action_part_number,human_action_part_name
0,0,834,"(Mises, 1996(Mises, [1949]], p. 1)",The practitioners of methodological holism are...,0,_4dGvzna,_97b4QYq,_dz5ekbM,96,47,...,367,43,Advances in Austrian Economics,100.000000,[],0,0,Chapter 0: Introduction,0,Part 0: Introduction
1,1,1842,"(Mises 2011, 1)","Interventionism, for its part, ""seeks to retai...",0,_yCU3SGE,_h8VDttU,_5k2NTcp,352,148,...,458,87,Quarterly Journal of Austrian Economics,100.000000,[],0,0,Chapter 0: Introduction,0,Part 0: Introduction
2,2,1197,Mises (1944: 1-19),"Indeed, Mises (1944: 1-19) explicitly cites th...",0,_ThHCvKg,_xKvBxdX,_EndqMMB,3399,1633,...,3855,337,Knowledge Management and Organizational Learning,76.923077,[],0,0,Chapter 0: Introduction,0,Part 0: Introduction
3,3,1197,"Mises, 1944: 1-19)","(Lippmann, 1938(Lippmann, /1943: 263;: 263; em...",3,_W7w3J4K,_FXcX2C6,_mpaD3bx,3456,1675,...,3855,337,Knowledge Management and Organizational Learning,76.923077,[],0,0,Chapter 0: Introduction,0,Part 0: Introduction
4,4,1920,"Mises's (1962, 1)","Indeed, the logical structure of the human min...",0,_JRV4vR4,_K4EEPwH,_kzagVzN,367,282,...,533,66,Review of Austrian Economics,100.000000,[],0,0,Chapter 0: Introduction,0,Part 0: Introduction


In [80]:
import pandas as pd

# Ler o CSV
df = pd.read_csv("keywords_austrian_papers.csv")

# Pegar a coluna, remover nulos, normalizar e obter únicos
unique_keywords_list = (
    df["Author Keywords"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .loc[lambda x: x != ""]   # remove strings vazias
    .unique()
)

# Converter para lista ordenada (opcional)
keywords = sorted(unique_keywords_list)

keywords = [kw for kw in keywords if kw.lower() not in {"mises", "hayek"}]

# Exibir resultado
print(len(keywords))
print(keywords)

2229
["'post-austrian' economics", '100% reserve banking', '100\xa0% reserves', '1920s', '2008 crisis', 'a priori knowledge', 'a00', 'a11', 'a12', 'abct', 'abstraction', 'academic freedom', 'accelerator', 'accountant', 'accounting', 'action', 'action axiom', 'action space', 'adam smith', 'adaptation', 'adaptive systems', 'additive political economy', 'adjustment', 'administrative law', 'administrative procedure act', 'adolf wagner', 'adr', 'adverse selection', 'aesthetics', 'afghanistan', 'agency costs', 'agency theory', 'agent based computational economics', 'agent-based computational economics', 'agent-based model', 'agent-based modeling', 'agglomeration', 'agglomeration economies', 'aggregate expenditure', 'aggregation', 'aggregation bias', 'agriculture', 'aid', 'akrasia', 'alchian', 'alertness', 'alfred marshall', 'alfred schutz', 'algorithms', 'altruism', 'ambiguity', 'american occupation of japan', 'amherst school', 'analytic narrative', 'analytic narratives', 'analytical anarchi

In [81]:
import re

def matched_keywords(text, keywords):
    if text is None:
        return ["NO-MATCH"]
    
    text = str(text).lower()
    
    matches = []
    
    for kw in keywords:
        kw_norm = kw.strip().lower()
        if not kw_norm:
            continue
        
        # Cria padrão com word boundaries
        pattern = r'\b' + re.escape(kw_norm) + r'\b'
        
        if re.search(pattern, text):
            matches.append(kw_norm)
    
    return matches if matches else ["NO-MATCH"]


# Testes
assert matched_keywords("This is Hayek", ["interventionism", "mises"]) == ["NO-MATCH"]
assert matched_keywords("This is interventionism", ["interventionism", "mises"]) == ["interventionism"]
assert matched_keywords("This is interventionism and Mises talked about it", ["interventionism", "mises"]) == ["interventionism", "mises"]

In [82]:
import pandas as pd
import re
from tqdm import tqdm

# Ativar integração do tqdm com pandas
tqdm.pandas()

def add_keywords_column(citations_df, keywords):
    citations_df["keywords"] = citations_df["context"].progress_apply(
        lambda text: matched_keywords(text, keywords)
    )
    return citations_df


mises_citations_df = add_keywords_column(mises_citations_df, keywords)
hayek_citations_df = add_keywords_column(hayek_citations_df, keywords)

# Visualizar
mises_citations_df[["context", "keywords"]].head()

100%|██████████| 6281/6281 [18:42<00:00,  5.60it/s]


,context,keywords
0,The practitioners of methodological holism are...,[NO-MATCH]
1,"Interventionism, for its part, ""seeks to retai...","[interventionism, private property, production]"
2,"Indeed, Mises (1944: 1-19) explicitly cites th...",[NO-MATCH]
3,"(Lippmann, 1938(Lippmann, /1943: 263;: 263; em...",[NO-MATCH]
4,"Indeed, the logical structure of the human min...","[epistemology, knowledge, power, structure]"


In [87]:
hayek_citations_df[["context", "keywords"]].head()

,context,keywords
0,"As Hayek (1931, chap. 1) himself emphasized, h...",[NO-MATCH]
1,"Hayek, more so perhaps than other Austrians, c...",[hume]
2,"-Friedrich Hayek, ""History and Politics"" (1954...","[accountant, friedrich hayek, knowledge, polit..."
3,"Hayek, more so perhaps than other Austrians, c...",[hume]
4,"Indeed, not only is collectivism not normal wi...",[NO-MATCH]


In [84]:
import pandas as pd

def keyword_conditional_table(mises_df, hayek_df):
    # ===== 1. Identificar sentence_ids =====
    mises_sentences = set(mises_df["sentence_id"].unique())
    hayek_sentences = set(hayek_df["sentence_id"].unique())
    
    # Partições
    mises_only_ids = mises_sentences - hayek_sentences
    hayek_only_ids = hayek_sentences - mises_sentences
    both_ids = mises_sentences & hayek_sentences

    # ===== 2. Função auxiliar =====
    def compute_counts_and_probs(df, valid_ids):
        sub = df[df["sentence_id"].isin(valid_ids)].copy()
        
        sub = sub.explode("keywords")
        sub = sub.dropna(subset=["keywords"])
        
        sub = sub.drop_duplicates(subset=["sentence_id", "keywords"])
        
        counts = sub.groupby("keywords")["sentence_id"].nunique()
        
        total_sentences = len(valid_ids)
        
        if total_sentences == 0:
            return pd.Series(dtype=float), pd.Series(dtype=float)
        
        probs = counts / total_sentences
        
        return counts, probs

    # ===== 3. Calcular =====
    c_mises_only, p_mises_only = compute_counts_and_probs(mises_df, mises_only_ids)
    c_hayek_only, p_hayek_only = compute_counts_and_probs(hayek_df, hayek_only_ids)
    
    combined_df = pd.concat([mises_df, hayek_df])
    c_both, p_both = compute_counts_and_probs(combined_df, both_ids)

    # ===== 4. Consolidar =====
    result = pd.concat(
        [
            c_mises_only.rename("count_mises_only"),
            p_mises_only.rename("P(keyword | Mises only)"),
            
            c_hayek_only.rename("count_hayek_only"),
            p_hayek_only.rename("P(keyword | Hayek only)"),
            
            c_both.rename("count_both"),
            p_both.rename("P(keyword | Mises AND Hayek)")
        ],
        axis=1
    ).fillna(0)

    # converter counts
    result["count_mises_only"] = result["count_mises_only"].astype(int)
    result["count_hayek_only"] = result["count_hayek_only"].astype(int)
    result["count_both"] = result["count_both"].astype(int)

    # ===== ✅ FILTRO NOVO =====
    result = result[
        (result["count_mises_only"] > 10) |
        (result["count_hayek_only"] > 10) |
        (result["count_both"] > 10)
    ]

    # ordenar
    result = result.sort_values(
        by="P(keyword | Mises AND Hayek)", ascending=False
    ).reset_index().rename(columns={"keywords": "keyword"})

    return result


# ===== USO =====
table = keyword_conditional_table(mises_citations_df, hayek_citations_df)

table = table.sort_values(by='count_both', ascending=False)

print(table.shape)

table.head(500)

(335, 7)


,keyword,count_mises_only,P(keyword | Mises only),count_hayek_only,P(keyword | Hayek only),count_both,P(keyword | Mises AND Hayek)
0,austrian,392,0.055745,260,0.046628,171,0.289340
1,von mises,1033,0.146900,61,0.010940,140,0.236887
2,kirzner,206,0.029295,135,0.024211,88,0.148900
3,process,258,0.036689,335,0.060079,59,0.099831
4,economics,422,0.060011,217,0.038917,59,0.099831
5,knowledge,206,0.029295,791,0.141858,55,0.093063
6,the market,364,0.051763,236,0.042324,49,0.082910
7,capital,255,0.036263,226,0.040531,48,0.081218
8,business cycle,74,0.010523,71,0.012733,46,0.077834
9,interest,199,0.028299,107,0.019189,46,0.077834


In [85]:
import numpy as np

def add_delta_split(df):
    df = df.copy()
    
    # garantir colunas
    cols = [
        "P(keyword | Mises only)",
        "P(keyword | Hayek only)",
        "P(keyword | Mises AND Hayek)"
    ]
    
    for c in cols:
        if c not in df.columns:
            df[c] = 0.0

    # deltas separados
    df["lift_mises"] = (
        df["P(keyword | Mises only)"] / df["P(keyword | Hayek only)"] 
    )
    
    df["lift_hayek"] = (
       df["P(keyword | Hayek only)"] / df["P(keyword | Mises only)"]  
    )

        
    df["lift_mises_hayek"] = (
        df["P(keyword | Mises AND Hayek)"] /
        df[["P(keyword | Mises only)", "P(keyword | Hayek only)"]].max(axis=1)
    )
    # ordenar pelo mais "co-citation driven"
    df = df.sort_values(by="lift_mises_hayek", ascending=False)

    return df


# ===== USO =====
table_with_deltas = add_delta_split(table)

table_with_deltas.to_csv("../data/processed/table_with_deltas.csv")

In [88]:
import pandas as pd

pd.set_option('display.max_rows', None)

table_with_deltas.sort_values(by='lift_mises_hayek', ascending=False)

,keyword,count_mises_only,P(keyword | Mises only),count_hayek_only,P(keyword | Hayek only),count_both,P(keyword | Mises AND Hayek),lift_mises,lift_hayek,lift_mises_hayek
72,disequilibrium,9,0.001280,6,0.001076,12,0.020305,1.189420,0.840746,15.864636
18,abct,16,0.002275,18,0.003228,29,0.049069,0.704841,1.418759,15.200602
56,socialist calculation debate,16,0.002275,11,0.001973,15,0.025381,1.153377,0.867019,11.154822
45,austrian business cycle,19,0.002702,18,0.003228,18,0.030457,0.836999,1.194744,9.434856
61,lavoie,20,0.002844,8,0.001435,14,0.023689,1.982366,0.504448,8.328934
43,austrian theory,29,0.004124,16,0.002869,19,0.032149,1.437216,0.695790,7.795554
68,austrian business cycle theory,19,0.002702,16,0.002869,13,0.021997,0.941624,1.061995,7.665821
51,calculation debate,27,0.003840,16,0.002869,17,0.028765,1.338097,0.747330,7.491634
57,socialist calculation,22,0.003129,20,0.003587,15,0.025381,0.872241,1.146472,7.076142
46,interest rates,32,0.004551,13,0.002331,18,0.030457,1.951868,0.512330,6.692893
